In [ ]:
from pathlib import Path

import rioxarray as rxr
import xrspatial as xrs

bathy = rxr.open_rasterio(Path("~/onnx_models/Coastwide_corrected.tif").expanduser())
bathy_band = bathy.where(bathy != bathy.rio.nodata).sel(band=1)  # Must preserve nodata as nan for slope calc

In [ ]:
# Create slope
slope = xrs.slope(bathy_band, method="planar", z_unit="meter", boundary="nan")

In [ ]:
slope = slope.fillna(85)
slope = slope.rio.write_nodata(85)

slope.rio.to_raster(
    Path("~/onnx_models/slope_10m_new.tif").expanduser(),
    driver="GTiff",
    compress="lzw",
    tiled=True,
    blockxsize=256,
    blockysize=256,
    interleave="pixel",
    photometric="MINISBLACK",
)

In [ ]:
# Fill bathy and slope NA values
bathy_band = bathy_band.fillna(-2000)
bathy_band = bathy_band.rio.write_nodata(-2000)

bathy_band.rio.to_raster(
    Path("~/onnx_models/bathymetry_10m_new.tif").expanduser(),
    driver="GTiff",
    compress="lzw",
    tiled=True,
    blockxsize=256,
    blockysize=256,
    interleave="pixel",
    photometric="MINISBLACK",
)